# Feature Engineering and Initial ML Model Testing

This script:
1. Loads cleaned Yelp review datasets from 'cleaned-data/'.
2. Converts text into TF-IDF features with n-grams.
3. Trains a basic Logistic Regression model for sentiment prediction.
4. Outputs basic accuracy metrics.

In [1]:
import glob
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression

import sys
sys.executable

'/opt/anaconda3/bin/python'

In [2]:
folder = "cleaned-data/"
csv_files = glob.glob(os.path.join(folder, "*.csv"))

dfs = []
for file in csv_files:
    df = pd.read_csv(file)
    df['state'] = os.path.splitext(os.path.basename(file))[0] 
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)
data = data.dropna()
print(f"Loaded {len(data)} reviews from {len(dfs)} states.")

Loaded 5222860 reviews from 19 states.


In [3]:
# creating sentiment score for easier classification
data['sentiment'] = data['stars'].apply(lambda x: 0 if x < 3 else (1 if x == 3 else 2))
data.sample(5)

,stars,text,review_length,num_exclamations,num_caps_words,clean_text,state,sentiment
3267833,5,"Great bartenders, hot chicks, decent DJ - grea...",21,0,1,great bartenders hot chicks decent dj great cr...,PA,2
809315,4,As always my favorite coffee spot in Reno!! Ev...,18,4,1,always favorite coffee spot reno every time im...,NV,2
2924723,5,Really Good. Make amazing steak egg and chees...,19,0,0,really good make amazing steak egg cheese brea...,PA,2
3211762,5,"This is my neighborhood pizza place, yes, ther...",54,0,0,neighborhood pizza place yes million area fami...,PA,2
1352738,5,Place a small i.e. cozy so not always easy to...,61,0,0,place small ie cozy notalways easy get table w...,TN,2


In [4]:
sample = data.sample(n=500000) # only using 500,000 data points for the sake of computation time (instead of 5,000,000) (arbitrary)

X = sample['clean_text']
# y = sample['stars'] # 1-5 RRP
y = sample['sentiment'] # pos/neg/neutral
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y)

# TF-IDF with n-grams
tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=10000)  # unigrams + bigrams
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Initial ML model: Logistic Regression 
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.87574

Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.86      0.85     10230
           1       0.57      0.37      0.45      5674
           2       0.92      0.96      0.94     34096

    accuracy                           0.88     50000
   macro avg       0.78      0.73      0.75     50000
weighted avg       0.86      0.88      0.87     50000



In [5]:
#Linear Regression model

y = sample['stars']    # 1–5
linreg = LinearRegression()
linreg.fit(X_train_tfidf, y_train)
pred = linreg.predict(X_test_tfidf)

mse = mean_squared_error(y_test, pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, pred)


print("MSE:", mse)
print("R2:", r2)
print("RMSE:",rmse)

# round predictions to nearest whole star
rounded_pred = np.round(pred)

# ensure predictions are between 1 and 5 (since regression might give 0.7 or 5.3)
rounded_pred = np.clip(rounded_pred, 1, 5)

# evaluate accuracy
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, rounded_pred)

print("Accuracy:", accuracy)


MSE: 0.1915915774974672
R2: 0.7091304677403554
RMSE: 0.437711751609969
Accuracy: 0.68168


In [8]:
# Multi-layered linear regression model

# Main Linear Regression Model
linreg = LinearRegression()
linreg.fit(X_train_tfidf, y_train)

main_pred = linreg.predict(X_test_tfidf)

# Baseline predictions (rounded)
baseline_pred = np.clip(np.round(main_pred), 1, 5)

# Ambiguous predictions between 2.5 and 3.5
ambiguous_mask = (main_pred > 2) & (main_pred < 4)


# 3-Star Specialist Classifier
# Build binary 3-star labels (must be done BEFORE train_test_split)
sample['is_three'] = (sample['stars'] == 3).astype(int)

# Align 3-star labels to training set
y_train_is_three = sample['is_three'].loc[y_train.index]

# train with logistic regression
three_clf = LogisticRegression(max_iter=1000)
three_clf.fit(X_train_tfidf, y_train_is_three)

three_probs = three_clf.predict_proba(X_test_tfidf)[:, 1]
is_three_pred = (three_probs > 0.3).astype(int)

# Final Prediction (Override ambiguous cases)
final_pred = np.where(is_three_pred == 1, 3, baseline_pred)
final_pred = np.clip(final_pred, 1, 5)

# Accuracy Metrics
accuracy = accuracy_score(y_test, final_pred)
baseline_accuracy = accuracy_score(y_test, baseline_pred)

print("Baseline (rounded linear regression) accuracy:", baseline_accuracy)
print("2-Stage model accuracy:", accuracy)

print("Number of ambiguous samples:", ambiguous_mask.sum())
print("Specialist predicts 3-stars in test:", is_three_pred.sum())
print("Both ambiguous AND predicted 3:", np.sum(ambiguous_mask & (is_three_pred == 1)))


Baseline (rounded linear regression) accuracy: 0.68168
2-Stage model accuracy: 0.6272
Number of ambiguous samples: 11427
Specialist predicts 3-stars in test: 5293
Both ambiguous AND predicted 3: 29


In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Use actual star ratings (1–5)
y = sample['stars']
X = sample['clean_text']

# Re-run train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y
)

# TF-IDF again (fit on training set only)
tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=10000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Multinomial logistic regression for 5 classes
clf = LogisticRegression(
    max_iter=300,
    multi_class='multinomial',
    solver='lbfgs',
    n_jobs=-1
)
clf.fit(X_train_tfidf, y_train)

pred = clf.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Accuracy: 0.67844
              precision    recall  f1-score   support

           1       0.73      0.82      0.77      6036
           2       0.50      0.37      0.43      4195
           3       0.52      0.44      0.48      5674
           4       0.55      0.48      0.51     11920
           5       0.77      0.87      0.82     22175

    accuracy                           0.68     50000
   macro avg       0.61      0.59      0.60     50000
weighted avg       0.66      0.68      0.67     50000



In [14]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

# Train Linear SVM
svm = LinearSVC(C=1.0)   # C=1 is a good default
svm.fit(X_train_tfidf, y_train)

# Predict on test set
svm_pred = svm.predict(X_test_tfidf)

# Accuracy
svm_accuracy = accuracy_score(y_test, svm_pred)
print("Linear SVM Accuracy:", svm_accuracy)

# Detailed metrics
print(classification_report(y_test, svm_pred))

Linear SVM Accuracy: 0.66856
              precision    recall  f1-score   support

           1       0.70      0.84      0.77      6036
           2       0.50      0.31      0.38      4195
           3       0.52      0.36      0.42      5674
           4       0.54      0.44      0.48     11920
           5       0.74      0.89      0.81     22175

    accuracy                           0.67     50000
   macro avg       0.60      0.57      0.57     50000
weighted avg       0.64      0.67      0.65     50000



In [15]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score

# ----------------------------
# 1. Define X and y
# ----------------------------

X = sample['clean_text']      # input text
y = sample['stars']           # TRUE star ratings (1–5)

# ----------------------------
# 2. Train/Test Split
# ----------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.1,
    random_state=42,
    stratify=y
)

# ----------------------------
# 3. TF-IDF Vectorization
# ----------------------------

tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=10000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# ----------------------------
# 4. Linear Regression Model
# ----------------------------

linreg = LinearRegression()
linreg.fit(X_train_tfidf, y_train)   # <-- y_train ARE the star ratings

# Predict continuous values (e.g. 3.27, 4.81, 2.02)
pred = linreg.predict(X_test_tfidf)

# ----------------------------
# 5. Regression metrics
# ----------------------------

mse = mean_squared_error(y_test, pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, pred)

print("MSE:", mse)
print("RMSE:", rmse)
print("R2:", r2)

# ----------------------------
# 6. Convert continuous predictions → star ratings
# ----------------------------

rounded_pred = np.round(pred)          # round to nearest star
rounded_pred = np.clip(rounded_pred, 1, 5)  # force into 1–5 range

# ----------------------------
# 7. Classification accuracy
# ----------------------------

accuracy = accuracy_score(y_test, rounded_pred)
print("Rounded Linear Regression Accuracy:", accuracy)


MSE: 0.54042321199192
RMSE: 0.7351348257237715
R2: 0.7212997410141897
Rounded Linear Regression Accuracy: 0.5499
